# 03 - Preprocessing
**Air Quality Prediction and Health Risk Analysis — Preprocessing**

This notebook prepares raw OpenAQ data for feature engineering and modelling. It is written for inclusion in an MSc dissertation appendix — the steps are documented and decisions justified.

**Goals:**
- Load raw files from `data/raw/`
- Assess data quality (missing values, duplicates, types, basic stats)
- Clean and transform data (timestamps, missing/value handling, outlier handling)
- Create analysis-ready dataset and save to `data/processed/clean_air_quality.csv`

## 1. Why preprocessing matters
Air quality measurement data often contain missing fields, duplicate readings, inconsistent timestamps, and extreme sensor values. Careful preprocessing reduces bias, improves model generalisation, and ensures reproducible results for the dissertation analysis.

## 2. Imports and configuration

In [2]:
# Standard libraries and project path discovery
from pathlib import Path
import sys
import json

# Data libraries
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # safe backend when running headless
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities for optional encoding / scaling
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set(style='whitegrid')

# Notebook-friendly display settings
pd.options.display.max_columns = 200
pd.options.display.float_format = '{:,.3f}'.format

In [3]:
# Robust project root discovery: prefer a folder that contains 'src' or 'requirements.txt'
def find_project_root(start: Path = Path.cwd(), max_up: int = 4) -> Path:
    p = start.resolve()
    for _ in range(max_up + 1):
        if (p / 'src').exists() or (p / 'requirements.txt').exists() or (p / 'pyproject.toml').exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    # fallback to cwd
    return start.resolve()

PROJECT_ROOT = find_project_root()
print('Project root:', PROJECT_ROOT)

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print('Raw dir:', RAW_DIR)
print('Processed dir:', PROCESSED_DIR)

Project root: C:\Users\phxac\Downloads\air-quality-forecasting
Raw dir: C:\Users\phxac\Downloads\air-quality-forecasting\data\raw
Processed dir: C:\Users\phxac\Downloads\air-quality-forecasting\data\processed


## 3. Load raw data
We load CSV(s) from `data/raw/`. The code below finds any CSV files and concatenates them into a single DataFrame. This approach is robust when ingestion produced multiple parts.

In [4]:
# Find CSV files in the raw directory
csv_files = sorted(RAW_DIR.glob('*.csv'))
if not csv_files:
    print('No CSV files found in', RAW_DIR)
    # show listing to help debug
    print(list(RAW_DIR.iterdir()) if RAW_DIR.exists() else 'RAW_DIR does not exist')
    df_raw = pd.DataFrame()
else:
    print('Found csv files:', [f.name for f in csv_files])
    # Read and concatenate; be conservative with dtypes
    frames = []
    for f in csv_files:
        try:
            df_part = pd.read_csv(f, low_memory=False)
            frames.append(df_part)
        except Exception as e:
            print('Failed to read', f, e)
    df_raw = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print('Raw shape:', df_raw.shape)
display(df_raw.head())
print('Columns:', list(df_raw.columns))

Found csv files: ['air_quality_data.csv']
Raw shape: (50, 10)


,location,city,country,latitude,longitude,parameter,value,unit,date_utc,raw_measurement
0,NaN,NaN,NaN,35.218,128.574,pm25,16.000,µg/m³,2026-08-04T09:00:00Z,"{""datetime"": {""utc"": ""2026-08-04T09:00:00Z"", ""..."
1,NaN,NaN,NaN,54.884,23.836,pm25,12.550,µg/m³,2026-08-04T09:00:00Z,"{""datetime"": {""utc"": ""2026-08-04T09:00:00Z"", ""..."
2,NaN,NaN,NaN,40.147,117.071,pm25,16.000,µg/m³,2021-08-09T11:00:00Z,"{""datetime"": {""utc"": ""2021-08-09T11:00:00Z"", ""..."
3,NaN,NaN,NaN,37.646,-118.967,pm25,9.000,µg/m³,2025-08-09T14:00:00Z,"{""datetime"": {""utc"": ""2025-08-09T14:00:00Z"", ""..."
4,NaN,NaN,NaN,38.592,-82.807,pm25,7.300,µg/m³,2026-08-04T09:00:00Z,"{""datetime"": {""utc"": ""2026-08-04T09:00:00Z"", ""..."


Columns: ['location', 'city', 'country', 'latitude', 'longitude', 'parameter', 'value', 'unit', 'date_utc', 'raw_measurement']


## 4. Data quality assessment
We look for missing data, duplicates, types, and basic statistics. Visual checks help identify issues for cleaning decisions.

In [5]:
# Basic info and missing values
if df_raw.empty:
    print('No raw data to inspect')
else:
    print('Data types:')
    display(df_raw.dtypes.astype(str))
    print('\nMissing values by column:')
    missing = df_raw.isna().sum().sort_values(ascending=False)
    display(missing[missing > 0])
    # Duplicates
    dup_count = df_raw.duplicated().sum()
    print(f'Full-row duplicates: {dup_count}')
    # Simple statistical summary for numeric columns
    display(df_raw.select_dtypes(include=[np.number]).describe().T)

Data types:


location           float64
city               float64
country            float64
latitude           float64
longitude          float64
parameter              str
value              float64
unit                   str
date_utc               str
raw_measurement        str
dtype: str


Missing values by column:


location    50
city        50
country     50
dtype: int64

Full-row duplicates: 0


,count,mean,std,min,25%,50%,75%,max
location,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,0.000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,50.000,28.097,24.611,-43.560,20.295,37.870,41.688,54.884
longitude,50.000,5.775,94.729,-122.986,-83.448,13.044,78.638,172.644
value,50.000,-179.487,"1,417.581","-9,999.000",6.075,9.792,18.255,238.000


In [7]:
# Missing values bar chart (top columns)
if not df_raw.empty:
    miss = df_raw.isna().sum().sort_values(ascending=False)
    top = miss[miss > 0].head(20)
    if len(top):
        fig, ax = plt.subplots(figsize=(8,4))
        top.plot.bar(ax=ax)
        ax.set_ylabel('Missing count')
        ax.set_title('Top missing-value columns')
        fig.tight_layout()
        fig_dir = PROJECT_ROOT / 'outputs' / 'figures'
        fig_dir.mkdir(parents=True, exist_ok=True)
        fig.savefig(fig_dir / 'missing_values_top.png')
        print('Saved missing-values figure')
    else:
        print('No missing values detected')

# Distribution plot for the measurement value column if present
if 'value' in df_raw.columns and not df_raw['value'].dropna().empty:
    fig = plt.figure(figsize=(6,4))
    sns.histplot(df_raw['value'].dropna(), bins=80, kde=False)
    plt.title('Distribution of measurement values')
    fig.savefig(PROJECT_ROOT / 'outputs' / 'figures' / 'value_distribution.png')
    print('Saved value_distribution.png')

Saved missing-values figure
Saved value_distribution.png


### Findings (placeholder)
Add a short description of what you observe from the missing-value chart and distributions. Example: `value` has many non-missing entries but `latitude`/`longitude` may be missing for some rows; duplicates may exist after concatenation.

## 5. Data cleaning steps
We implement conservative cleaning: remove obvious duplicates, convert timestamps, drop rows that cannot be used for modelling (no `value`), and handle implausible values. Each decision is explained.

In [8]:
df = df_raw.copy()
if df.empty:
    print('No data to clean; stopping here')
else:
    # 1. Drop exact duplicates
    before = len(df)
    df = df.drop_duplicates(ignore_index=True)
    print(f'Dropped {before - len(df)} exact duplicate rows')

    # 2. Ensure timestamp column exists and convert
    # Common raw column name is 'date_utc' from ingestion. Try several candidate names.
    date_cols = [c for c in df.columns if 'date' in c.lower()]
    date_col = None
    for c in date_cols:
        if 'utc' in c.lower() or 'iso' in c.lower() or 'timestamp' in c.lower():
            date_col = c
            break
    if date_col is None and date_cols:
        date_col = date_cols[0]
    if date_col is not None:
        df['date_utc'] = pd.to_datetime(df[date_col], utc=True, errors='coerce')
        n_bad_dates = df['date_utc'].isna().sum()
        print(f'Parsed dates; {n_bad_dates} rows have invalid timestamps')
    else:
        print('No date-like column found; creating placeholder date_utc with NaT')
        df['date_utc'] = pd.NaT

    # 3. Drop rows without measurement 'value' (cannot be used for supervised regression)
    if 'value' in df.columns:
        before = len(df)
        df = df[~df['value'].isna()].copy()
        print(f'Dropped {before - len(df)} rows missing measurement value')
    else:
        print('No value column present — check ingestion format')

    # 4. Cast numeric columns where appropriate
    numeric_candidates = ['value', 'latitude', 'longitude']
    for col in numeric_candidates:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # 5. Remove impossible values: negative measurement values unlikely for pollutant concentration (but 0 allowed)
    if 'value' in df.columns:
        n_neg = (df['value'] < 0).sum()
        if n_neg:
            print(f'Removing {n_neg} rows with negative measurement values')
            df = df[df['value'] >= 0].copy()

    # 6. Sort chronologically when possible
    if 'date_utc' in df.columns:
        df = df.sort_values('date_utc').reset_index(drop=True)

    print('Cleaned shape:', df.shape)

Dropped 0 exact duplicate rows
Parsed dates; 0 rows have invalid timestamps
Dropped 0 rows missing measurement value
Removing 1 rows with negative measurement values
Cleaned shape: (49, 10)


### Outlier handling rationale
Air pollutant measurements can have heavy tails. We use a parameter-wise IQR filter to remove extreme outliers (conservative) — this reduces the influence of sensor spikes while preserving valid extreme pollution events when reasonable.

In [9]:
# Outlier removal per pollutant `parameter` using IQR (1.5*IQR rule)
if not df.empty and 'parameter' in df.columns and 'value' in df.columns:
    def remove_outliers_iqr(group, k=1.5):
        q1 = group['value'].quantile(0.25)
        q3 = group['value'].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - k * iqr
        upper = q3 + k * iqr
        return group[(group['value'] >= lower) & (group['value'] <= upper)]

    before = len(df)
    df = df.groupby('parameter', group_keys=False).apply(remove_outliers_iqr).reset_index(drop=True)
    after = len(df)
    print(f'Removed {before - after} rows as parameter-wise IQR outliers')
else:
    print('Skipping outlier removal (missing columns)')

Removed 4 rows as parameter-wise IQR outliers


## 6. Feature transformations
Extract time features from `date_utc` and encode categorical variables conservatively for later feature engineering. Scaling is left optional — explained below.

In [11]:
if df.empty:
    print('No data to transform')
else:
    # Time features
    df['year'] = df['date_utc'].dt.year
    df['month'] = df['date_utc'].dt.month
    df['day'] = df['date_utc'].dt.day
    df['hour'] = df['date_utc'].dt.hour
    df['day_of_week'] = df['date_utc'].dt.dayofweek

    # Categorical columns to keep: parameter, location, city, country
    categorical_cols = [c for c in ['parameter', 'location', 'city', 'country'] if c in df.columns]
    print('Categorical columns:', categorical_cols)

    # For reproducibility we will encode `parameter` using one-hot encoding and keep location/city as-is (they may be high-cardinality).
    if 'parameter' in df.columns:
        dummies = pd.get_dummies(df['parameter'], prefix='param')
        df = pd.concat([df, dummies], axis=1)

    # Optional scaling flag: scaling often unnecessary for tree models, useful for linear models / distance-based models
    scale_numeric = False  # change to True if you're planning to use linear/kNN/SVM models
    numeric_cols = [c for c in ['value'] if c in df.columns]
    if scale_numeric and numeric_cols:
        scaler = StandardScaler()
        df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
        # Save scaler to disk if you need to apply same transform in production
        import joblib
        joblib.dump(scaler, PROCESSED_DIR / 'scaler.joblib')
        print('Saved StandardScaler to processed dir')

    print('Transformed shape:', df.shape)

Categorical columns: ['location', 'city', 'country']
Transformed shape: (45, 14)


## 7. Save processed dataset
We save a clean CSV to `data/processed/clean_air_quality.csv`. This file will be the input to the feature engineering notebook.

In [12]:
OUT_PATH = PROCESSED_DIR / 'clean_air_quality.csv'
if df.empty:
    print('No cleaned data to save')
else:
    # Final checks
    print('Final shape before save:', df.shape)
    print('Remaining missing values (per column):')
    display(df.isna().sum()[lambda x: x>0])
    # Save CSV without the dataframe index for portability
    df.to_csv(OUT_PATH, index=False)
    print('Saved cleaned dataset to', OUT_PATH)

Final shape before save: (45, 14)
Remaining missing values (per column):


location    45
city        45
country     45
dtype: int64

Saved cleaned dataset to C:\Users\phxac\Downloads\air-quality-forecasting\data\processed\clean_air_quality.csv


## 8. Summary & next steps
- Duplicates removed, timestamps parsed, negative measurement values removed, and conservative outlier filtering applied per pollutant.
- Time-based features (`year`, `month`, `day`, `hour`, `day_of_week`) were extracted.
- `parameter` was one-hot encoded; other categorical fields retained for later hierarchical feature engineering.
- Scaling is optional and saved if enabled.

Next: feature engineering (rolling means, lag features, meteorological joins) in `04_feature_engineering.ipynb`.